# Threshold Calibration from Operator Feedback

**HH Goa 2026 — Task #3**

Every time you mark a returned match **correct** or **incorrect** in the UI, the
distance that produced it is appended to `backend/data/feedback.jsonl`. This
notebook turns those labels into a defensible decision threshold.

---

## What this does and does not do

**Does:** learn the best **decision boundary** (the L2 threshold) for *your*
photos, camera and lighting, and show the error you should expect at it.

**Does not:** retrain SFace. Fine-tuning a face-recognition network needs tens
of thousands of labelled identities and a GPU; a few dozen feedback clicks
cannot move those weights. The embedding stays fixed — what improves with your
labels is where the line is drawn through it, and per-deployment that is the
part that actually matters.

**No images or embeddings are stored.** Only distances, labels and provenance.

In [ ]:
import json, os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

BACKEND = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BACKEND))

from app.services.face_processor import SFACE_L2_THRESHOLD, SFACE_COSINE_THRESHOLD

FEEDBACK = BACKEND / "data" / "feedback.jsonl"
print("feedback file :", FEEDBACK)
print("exists        :", FEEDBACK.exists())
print("published L2  :", SFACE_L2_THRESHOLD, "| cosine:", SFACE_COSINE_THRESHOLD)

## 1. Load the labels

In [ ]:
rows = []
if FEEDBACK.exists():
    for line in FEEDBACK.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass

same = np.array([r["euclidean_distance"] for r in rows if r.get("label") == "correct"])
diff = np.array([r["euclidean_distance"] for r in rows if r.get("label") == "incorrect"])

print(f"labels total      : {len(rows)}")
print(f"  same person     : {len(same)}")
print(f"  different people: {len(diff)}")
print(f"  unsure          : {sum(1 for r in rows if r.get('label') == 'unsure')}")

if len(same) == 0 or len(diff) == 0:
    print("\nNeed at least one of EACH label before anything can be calibrated.")
    print("Run some searches and mark the results in the UI, then re-run this notebook.")

## 2. Are the two classes actually separable?

If the histograms overlap heavily, **no threshold will work well** — that points
at input quality (lighting, pose, resolution), not at the number.

In [ ]:
if len(same) and len(diff):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bins = np.linspace(0, 1.6, 33)
    ax.hist(same, bins=bins, alpha=0.65, label=f"same person (n={len(same)})", color="#4a9d5f")
    ax.hist(diff, bins=bins, alpha=0.65, label=f"different people (n={len(diff)})", color="#c25b5b")
    ax.axvline(SFACE_L2_THRESHOLD, color="#333", ls="--",
               label=f"published default ({SFACE_L2_THRESHOLD})")
    ax.set_xlabel("L2 distance"); ax.set_ylabel("count")
    ax.set_title("Score distribution by operator label")
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    gap = diff.min() - same.max()
    print(f"same person   : min {same.min():.4f}  max {same.max():.4f}  mean {same.mean():.4f}")
    print(f"different     : min {diff.min():.4f}  max {diff.max():.4f}  mean {diff.mean():.4f}")
    print(f"\nseparation gap: {gap:+.4f}", "(clean split)" if gap > 0 else "(classes OVERLAP)")

## 3. Sweep every threshold

For each candidate cut-off, compute:

- **TPR** — genuine matches accepted (higher is better)
- **TNR** — impostors rejected (higher is better)
- **Balanced accuracy** — their mean, which is robust to unequal class counts

In [ ]:
if len(same) and len(diff):
    ts = np.arange(0.20, 1.60, 0.005)
    tpr = np.array([(same <= t).mean() for t in ts])
    tnr = np.array([(diff > t).mean() for t in ts])
    bal = (tpr + tnr) / 2

    best_i = int(bal.argmax())
    best_t = float(ts[best_i])

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(ts, tpr, label="TPR — genuine accepted", color="#4a9d5f")
    ax.plot(ts, tnr, label="TNR — impostors rejected", color="#c25b5b")
    ax.plot(ts, bal, label="balanced accuracy", color="#2f6fb0", lw=2.5)
    ax.axvline(SFACE_L2_THRESHOLD, color="#333", ls="--", label=f"default ({SFACE_L2_THRESHOLD})")
    ax.axvline(best_t, color="#e0a020", ls=":", lw=2, label=f"best here ({best_t:.3f})")
    ax.set_xlabel("L2 threshold"); ax.set_ylabel("rate"); ax.set_ylim(-0.02, 1.02)
    ax.set_title("Threshold sweep"); ax.legend(loc="lower left"); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    at_default = float(bal[np.abs(ts - SFACE_L2_THRESHOLD).argmin()])
    print(f"best threshold      : {best_t:.4f}   balanced accuracy {bal[best_i]:.4f}")
    print(f"published default   : {SFACE_L2_THRESHOLD}   balanced accuracy {at_default:.4f}")
    print(f"improvement         : {bal[best_i] - at_default:+.4f}")

## 4. Confusion matrix at each threshold

In [ ]:
def confusion(t):
    tp = int((same <= t).sum()); fn = int((same > t).sum())
    fp = int((diff <= t).sum()); tn = int((diff > t).sum())
    total = tp + fn + fp + tn
    return {"threshold": round(t, 4), "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "accuracy": round((tp + tn) / total, 4) if total else None}

if len(same) and len(diff):
    for label, t in (("published default", SFACE_L2_THRESHOLD), ("best from your labels", best_t)):
        c = confusion(t)
        print(f"{label:<22} t={c['threshold']:.4f}  "
              f"TP={c['TP']} FN={c['FN']} FP={c['FP']} TN={c['TN']}  acc={c['accuracy']}")
    print("\nFN = a real match you would miss.   FP = a stranger accepted as a match.")

## 5. Verdict

Read this before changing anything in production.

In [ ]:
if len(same) and len(diff):
    enough = len(same) >= 10 and len(diff) >= 10
    delta = bal[best_i] - at_default

    print(f"samples: {len(same)} same-person, {len(diff)} different-person\n")
    if not enough:
        print("NOT ENOUGH DATA YET.")
        print("  Under ~10 of each class the suggestion swings with a single new label.")
        print(f"  Keep the published default of {SFACE_L2_THRESHOLD} and keep labelling.")
    elif delta < 0.02:
        print("KEEP THE DEFAULT.")
        print(f"  Your data agrees with {SFACE_L2_THRESHOLD} (gain only {delta:+.4f}).")
    else:
        print(f"CONSIDER {best_t:.3f}  (balanced accuracy {delta:+.4f} over the default)")
        print("  Set it in the UI slider, or send `threshold` in the API call.")
        print("  It is recorded in the on-chain canonical record, so the change stays auditable.")
else:
    print("No labels yet — mark some results in the UI first.")

---

## How to collect labels

1. Run a search in the UI (tab **01**) or via `run_pipeline.py`.
2. For each returned match press **Correct** or **Not this person**.
3. Do the same for results you *know* are wrong — impostor labels matter just as
   much as genuine ones. A model calibrated only on matches will happily accept
   everybody.
4. Aim for **10+ of each** before trusting the suggestion.
5. Re-run this notebook.

The same numbers are available without Jupyter at
`GET /api/social/feedback/stats`.